# RAG Pipeline — Step-by-Step Walkthrough
### Corpus: Climate Change in India

---

## What is RAG? (The Big Picture)

A language model like GPT-4 knows a lot — but only what it saw during training. Ask it about a document it never read, a recent event, or your private data, and it will either say it doesn't know or, worse, make something up (hallucinate).

**RAG fixes this by giving the model a library card.**

Instead of relying on memory alone, the model is allowed to *look things up* before answering. The pipeline:
1. Stores your documents in a searchable database
2. When a question arrives, finds the most relevant passages
3. Hands those passages to the LLM as context
4. The LLM reads the passages and writes a grounded answer

```
Your Documents
     │
     ▼
[1] LOAD   →  raw text
[2] CHUNK  →  small passages
[3] EMBED  →  vectors (numbers that represent meaning)
[4] STORE  →  vector database (searchable index)
                    │
               User Question
                    │
[5] RETRIEVE  →  find the most relevant passages
[6] GENERATE  →  LLM reads passages → writes answer
```

We'll run each of these 6 steps one at a time below.

---
## Setup — Configuration

We configure the pipeline once here. Everything below reads from `config`.

- **LLM:** `gpt-4o-mini` — generates the final answer
- **Embeddings:** `text-embedding-3-small` — converts text to vectors
- **Vector store:** ChromaDB (local, persisted to `./chroma_db`)

In [1]:
import shutil, os
from dotenv import load_dotenv
load_dotenv()

# Fresh start — wipe any previous index
if os.path.exists('./chroma_db'):
    shutil.rmtree('./chroma_db')
    print('Cleared old index.')

from rag.config import RAGConfig

config = RAGConfig(
    llm_provider='openai',
    embedding_provider='openai',
    chunk_size=1000,
    chunk_overlap=150,
    top_k=5,
    search_type='mmr',
)

print(f'LLM:        {config.llm_provider} / {config.llm_model}')
print(f'Embeddings: {config.embedding_provider} / {config.embedding_model}')
print(f'Chunk size: {config.chunk_size} chars  |  Overlap: {config.chunk_overlap} chars')
print(f'Retrieval:  top_k={config.top_k}, strategy={config.search_type}')

LLM:        openai / gpt-4o-mini
Embeddings: openai / text-embedding-3-small
Chunk size: 1000 chars  |  Overlap: 150 chars
Retrieval:  top_k=5, strategy=mmr


---
## Step 1 — Document Loading

**What happens here:** The loader fetches each URL, strips HTML tags, and returns a list of `Document` objects. Each `Document` has:
- `.page_content` — the raw text
- `.metadata` — where it came from (URL, title, etc.)

**Mental model:** Imagine photocopying 4 articles from different websites. You now have a stack of plain-text pages. The pipeline doesn't care they came from Wikipedia — they're just text now.

**Why we load from 4 sources:** A single article gives you one perspective. Four articles on related topics (the problem, the climate baseline, the energy response, the deforestation angle) give the retriever more surface area to find relevant passages for diverse questions.

In [3]:
from rag.ingest import load_documents

URLS = [
    'https://en.wikipedia.org/wiki/Climate_change_in_India',
    'https://en.wikipedia.org/wiki/Climate_of_India',
    'https://en.wikipedia.org/wiki/Renewable_energy_in_India',
    'https://en.wikipedia.org/wiki/Deforestation_in_India',
]

print('Loading documents from Wikipedia...')
raw_docs = load_documents(web_urls=URLS)

print(f'\nLoaded {len(raw_docs)} documents\n')
for doc in raw_docs:
    source = doc.metadata.get('source', '?')
    words  = len(doc.page_content.split())
    chars  = len(doc.page_content)
    print(f'  {source[-50:]:50s}  {words:>5,} words  ({chars:>6,} chars)')

total_words = sum(len(d.page_content.split()) for d in raw_docs)
print(f'\nTotal: {total_words:,} words across all documents')
print('\n--- Sample text (first 500 chars of document 1) ---')
print(raw_docs[0].page_content[:500])

Loading documents from Wikipedia...
  Loading 4 web URL(s)...

Loaded 4 documents

  ps://en.wikipedia.org/wiki/Climate_change_in_India  10,826 words  (76,688 chars)
  https://en.wikipedia.org/wiki/Climate_of_India      13,683 words  (90,371 chars)
  ://en.wikipedia.org/wiki/Renewable_energy_in_India  9,125 words  (61,795 chars)
  tps://en.wikipedia.org/wiki/Deforestation_in_India  2,020 words  (15,465 chars)

Total: 35,654 words across all documents

--- Sample text (first 500 chars of document 1) ---




Climate change in India - Wikipedia






























Jump to content







Main menu





Main menu
move to sidebar
hide



		Navigation
	


Main pageContentsCurrent eventsRandom articleAbout WikipediaContact us





		Contribute
	


HelpLearn to editCommunity portalRecent changesUpload fileSpecial pages



















Search











Search






















Appearance
















Donate

Create account

Log in








Personal tools






Donate


Create

---
## Step 2 — Chunking

**The problem:** Each document is thousands of words. We can't hand an entire Wikipedia article to the LLM as context — it would be too long, too expensive, and would dilute the relevant parts. We also can't embed a whole article as one vector, because a single vector can't capture every topic it covers.

**The solution:** Cut the documents into small, overlapping passages called *chunks*.

**Mental model:** Imagine cutting a newspaper article into index cards, one paragraph per card. If a question is about "solar energy in Rajasthan", you only need the 3 cards that mention that — not all 40 cards from the article.

**Why overlap?** The last sentence of one chunk might be the setup for the first sentence of the next. A 150-character overlap means each chunk shares a bit of context with its neighbours, so we don't lose meaning at the cut points.

**Parameters we're using:**
- `chunk_size = 1000` characters (~150–200 words per chunk)
- `chunk_overlap = 150` characters (~1–2 sentences of shared context)

In [4]:
from rag.ingest import chunk_documents

chunks = chunk_documents(
    raw_docs,
    chunk_size=config.chunk_size,
    chunk_overlap=config.chunk_overlap,
)

print(f'Documents:  {len(raw_docs)}')
print(f'Chunks:     {len(chunks)}')
print(f'Avg chunk:  {sum(len(c.page_content) for c in chunks) // len(chunks)} chars')

# Show chunk size distribution
sizes = [len(c.page_content) for c in chunks]
print(f'Smallest:   {min(sizes)} chars')
print(f'Largest:    {max(sizes)} chars')

# Show two consecutive chunks so the overlap is visible
print('\n--- Chunk 10 ---')
print(chunks[10].page_content)
print('\n--- Chunk 11 (notice how it begins — overlaps with end of chunk 10) ---')
print(chunks[11].page_content)

# Show that the overlap is real
end_of_10   = chunks[10].page_content[-150:]
start_of_11 = chunks[11].page_content[:150]
print(f'\n--- Overlap check ---')
print(f'End of chunk 10:     ...{end_of_10}')
print(f'Start of chunk 11:   {start_of_11}...')

Documents:  4
Chunks:     348
Avg chunk:  731 chars
Smallest:   5 chars
Largest:    998 chars

--- Chunk 10 ---
Industry[edit]
The industrial sector, including the production of cement,[29] iron, and steel,[30] is a major contributor to global emissions, accounting for about a quarter of the total.[31] From 2000 to 2014, fuel consumption in this sector surged by 406%,[23] reflecting its rapid expansion and escalating energy demand. By 2014, the industry was also responsible for 42% of total energy consumption,[23] highlighting the significant environmental effects associated with industrial activities.

--- Chunk 11 (notice how it begins — overlaps with end of chunk 10) ---
Industry-related emissions make up half of India's carbon emissions and are increasing.[20] India, the world's second-largest steel producer, accounts for 7% of global crude steel production. With rising domestic demand, emissions from this sector could significantly increase. Implementing standards for low and near

---
## Step 3 — Embeddings

**What is an embedding?** An embedding is a list of numbers (a vector) that represents the *meaning* of a piece of text. Texts with similar meaning end up with similar vectors — they are close to each other in "meaning space".

**Mental model:** Imagine a map where every sentence is plotted as a point. Sentences about "monsoon rainfall" cluster near each other. Sentences about "solar panels" cluster elsewhere. An embedding is just the GPS coordinates of a sentence on that map.

- `text-embedding-3-small` produces **1536-dimensional** vectors — 1536 numbers per chunk.
- That's 1536 axes in meaning-space. Each axis captures some aspect of meaning (topic, sentiment, domain, etc.) — we can't directly interpret individual dimensions, but the overall geometry is meaningful.

**Why does this work?** Because the embedding model was trained on billions of text pairs. It learned that "India" and "subcontinent" are close, that "deforestation" and "carbon sink" are related, that "monsoon" and "rainfall" cluster together.

In the next cell we'll embed a single chunk and look at the actual numbers.

In [5]:
from rag.ingest import build_embeddings
import math

embedding_model = build_embeddings(
    provider=config.embedding_provider,
    model_name=config.embedding_model,
    api_key=config.openai_api_key,
)

# Embed a single chunk and look at the raw vector
sample_chunk = chunks[10].page_content
print(f'Text being embedded ({len(sample_chunk)} chars):')
print(sample_chunk[:200] + '...')
print()

vector = embedding_model.embed_query(sample_chunk)

print(f'Vector dimensions: {len(vector)}')
print(f'First 10 values:   {[round(v, 4) for v in vector[:10]]}')
print(f'Magnitude (length): {round(math.sqrt(sum(v**2 for v in vector)), 4)}')

# Now show that semantically similar texts have similar vectors
text_a = 'India faces severe flooding due to climate change'
text_b = 'Rising sea levels and extreme rainfall threaten Indian coastal cities'
text_c = 'Solar panel installation has grown rapidly in Rajasthan'

vec_a = embedding_model.embed_query(text_a)
vec_b = embedding_model.embed_query(text_b)
vec_c = embedding_model.embed_query(text_c)

def cosine_similarity(v1, v2):
    dot   = sum(a * b for a, b in zip(v1, v2))
    mag1  = math.sqrt(sum(a**2 for a in v1))
    mag2  = math.sqrt(sum(b**2 for b in v2))
    return dot / (mag1 * mag2)

sim_ab = cosine_similarity(vec_a, vec_b)
sim_ac = cosine_similarity(vec_a, vec_c)

print(f'\n--- Semantic similarity demo ---')
print(f'A: "{text_a}"')
print(f'B: "{text_b}"')
print(f'C: "{text_c}"')
print(f'\nSimilarity A<->B (same topic — flooding/climate): {sim_ab:.4f}')
print(f'Similarity A<->C (different topic — solar):       {sim_ac:.4f}')
print(f'\nA and B are more similar ({sim_ab:.4f} > {sim_ac:.4f}) — the embedding captures meaning, not just keywords.')

Text being embedded (497 chars):
Industry[edit]
The industrial sector, including the production of cement,[29] iron, and steel,[30] is a major contributor to global emissions, accounting for about a quarter of the total.[31] From 200...

Vector dimensions: 1536
First 10 values:   [0.0111, 0.0015, 0.0599, -0.0084, 0.0208, 0.0187, 0.0046, 0.0498, 0.0136, -0.0044]
Magnitude (length): 1.0001

--- Semantic similarity demo ---
A: "India faces severe flooding due to climate change"
B: "Rising sea levels and extreme rainfall threaten Indian coastal cities"
C: "Solar panel installation has grown rapidly in Rajasthan"

Similarity A<->B (same topic — flooding/climate): 0.6694
Similarity A<->C (different topic — solar):       0.2522

A and B are more similar (0.6694 > 0.2522) — the embedding captures meaning, not just keywords.


---
## Step 4 — Vector Store (Indexing)

**What is a vector store?** A database that stores your chunks alongside their vectors, and can quickly find the chunks whose vectors are closest to a query vector.

**Mental model:** Going back to the map analogy — the vector store is the map itself. Each chunk is a pin on the map. When you ask a question, the question gets its own pin, and the vector store finds the 5 nearest pins. Those nearest pins are the most relevant chunks.

**Why ChromaDB?** It runs locally (no server needed), persists to disk, and is fast enough for corpora up to ~100K chunks. At larger scale you'd use Pinecone, Weaviate, or pgvector.

**What happens on disk:** ChromaDB stores the vectors in a binary format (HNSW index — a graph structure that enables fast approximate nearest-neighbour search). After this cell you'll see a `chroma_db/` folder appear.

In [6]:
from rag.ingest import get_vectorstore

print('Building vector store...')
print(f'Embedding {len(chunks)} chunks — this calls the OpenAI API once per batch.')
print('(OpenAI batches up to 2048 texts per API call, so this is efficient.)\n')

vectorstore = get_vectorstore(
    persist_dir=config.chroma_persist_dir,
    collection_name=config.collection_name,
    embeddings=embedding_model,
)
vectorstore.add_documents(chunks)

total = vectorstore._collection.count()
print(f'Index built. {total} chunks stored in ChromaDB.')
print(f'Each chunk is stored with its {len(vector)}-dimensional vector.')
print(f'Total vectors in DB: {total} x {len(vector)} = {total * len(vector):,} numbers on disk.')

# Peek at what's in the store
sample = vectorstore._collection.get(limit=3)
print('\n--- 3 randomly sampled stored chunks ---')
for text, meta in zip(sample['documents'], sample['metadatas']):
    source = meta.get('source', '?')[-40:]
    print(f'  [{source}] {text[:100]}...')

Building vector store...
Embedding 348 chunks — this calls the OpenAI API once per batch.
(OpenAI batches up to 2048 texts per API call, so this is efficient.)

Index built. 348 chunks stored in ChromaDB.
Each chunk is stored with its 1536-dimensional vector.
Total vectors in DB: 348 x 1536 = 534,528 numbers on disk.

--- 3 randomly sampled stored chunks ---
  [kipedia.org/wiki/Climate_change_in_India] Climate change in India - Wikipedia






























Jump to content







Main menu

...
  [kipedia.org/wiki/Climate_change_in_India] Toggle Effects on the natural environment subsection





2.1
Temperature and weather changes






...
  [kipedia.org/wiki/Climate_change_in_India] 6.2
Activism










7
See also








8
References




Toggle References subsection





8.1
Bib...


---
## Step 5 — Retrieval

**What happens here:** The user's question is embedded into a vector (same model, same space as the chunks). Then the vector store finds the K chunks whose vectors are closest — "closest" meaning most semantically similar.

**MMR vs. plain similarity search:**
- **Similarity search** returns the K most similar chunks. Problem: if 3 chunks are near-identical (say, three paragraphs all saying "India's temperature is rising"), you waste context window.
- **MMR (Maximal Marginal Relevance)** picks the next chunk that is *both relevant to the query AND different from what's already selected*. You get diversity + relevance.

**Mental model:** Similarity search is like Google returning 5 results from the same webpage. MMR is like Google returning results from 5 different, complementary sources.

In the next cell we'll run retrieval and inspect exactly which chunks come back — *before* the LLM ever sees them.

In [7]:
from rag.retriever import get_retriever

retriever = get_retriever(
    vectorstore=vectorstore,
    search_type=config.search_type,
    k=config.top_k,
)

question = 'What are the main impacts of climate change on India\'s agriculture and food security?'

print(f'Question: "{question}"')
print(f'\nRetrieving top-{config.top_k} chunks using {config.search_type.upper()}...\n')

retrieved = retriever.invoke(question)

print(f'Retrieved {len(retrieved)} chunks:\n')
for i, doc in enumerate(retrieved, 1):
    source  = doc.metadata.get('source', '?')[-45:]
    chars   = len(doc.page_content)
    preview = doc.page_content[:200].replace('\n', ' ')
    print(f'[{i}] Source: {source}')
    print(f'    Length: {chars} chars')
    print(f'    Text:   {preview}...')
    print()

print('--- This is the EXACT text the LLM will see as context. No more, no less. ---')

Question: "What are the main impacts of climate change on India's agriculture and food security?"

Retrieving top-5 chunks using MMR...

Retrieved 5 chunks:

[1] Source: en.wikipedia.org/wiki/Climate_change_in_India
    Length: 496 chars
    Text:   Reduced crop yields[edit] Climate change in India will have disproportionate ramifications for the more than 400 million that make up India's poor community. This is because so many depend on natural ...

[2] Source: en.wikipedia.org/wiki/Climate_change_in_India
    Length: 610 chars
    Text:   Mitigation[edit] India is ranked high in Climate Change Performance Index 2021  Greenhouse gas sinks[edit] Land use, land-use change, and forestry absorbed 300 Mt of CO2eq in 2014[30] and in 2020 tota...

[3] Source: en.wikipedia.org/wiki/Climate_change_in_India
    Length: 699 chars
    Text:   Temperature rises on the Tibetan Plateau are causing Himalayan glaciers to retreat, threatening the flow rate of the Ganges, Brahmaputra, Yamuna and other m

### Under the hood — query vector vs. chunk vectors

At retrieval time, only **one** embedding API call is made — for your question. The chunk vectors were already stored in Step 4 and ChromaDB reads them directly from disk.

The cell below re-embeds the retrieved chunks **purely for display** so we can print the similarity scores and make the math visible. This is not part of the real pipeline.

```
What actually happens at retrieval time:
  1. Embed the question  →  query vector  (1 API call)
  2. ChromaDB reads stored chunk vectors from disk  (no API call)
  3. Compute cosine similarity between query and each chunk
  4. Return top-K closest chunks
```

In [8]:
import math

def cosine_similarity(v1, v2):
    dot  = sum(a * b for a, b in zip(v1, v2))
    mag1 = math.sqrt(sum(a**2 for a in v1))
    mag2 = math.sqrt(sum(b**2 for b in v2))
    return dot / (mag1 * mag2)

def bar(score, width=30):
    filled = round(score * width)
    return '[' + '#' * filled + '-' * (width - filled) + ']'

# Embed the question — this is the one real API call retrieval makes
print('Embedding the question...')
query_vector = embedding_model.embed_query(question)
print(f'  Query vector: {len(query_vector)} dimensions')
print(f'  First 5 values: {[round(v, 4) for v in query_vector[:5]]}')

# Re-embed retrieved chunks FOR DISPLAY ONLY.
# ChromaDB already stored these vectors in Step 4 and used them internally during retrieval.
# We call embed_documents() here only so we can print the similarity scores below.
print(f'\n[Display only] Re-embedding {len(retrieved)} chunks to show similarity scores...')
print('(In the real pipeline this step does not happen — ChromaDB reads from disk.)\n')
chunk_texts   = [doc.page_content for doc in retrieved]
chunk_vectors = embedding_model.embed_documents(chunk_texts)

print(f'Query: "{question[:80]}..."')
print()
print(f'  {"Rank":<5} {"Cosine Sim":>10}  {"Visual":35}  Source')
print(f'  {"-"*4} {"-"*10}  {"-"*35}  {"-"*40}')

for i, (doc, vec) in enumerate(zip(retrieved, chunk_vectors), 1):
    sim    = cosine_similarity(query_vector, vec)
    source = doc.metadata.get('source', '?')[-40:]
    print(f'  [{i}]   {sim:.4f}    {bar(sim)}  {source}')

print()
print('Scores range from 0 (completely unrelated) to 1 (identical meaning).')
print('Scores > 0.75 generally indicate strong topical relevance for this embedding model.')
print()
print('Note: MMR already ran internally and chose DIVERSE chunks. If we had used plain')
print('similarity search, the top 5 scores would be higher but possibly redundant.')

Embedding the question...
  Query vector: 1536 dimensions
  First 5 values: [0.0155, 0.0232, 0.0872, 0.0021, -0.0283]

[Display only] Re-embedding 5 chunks to show similarity scores...
(In the real pipeline this step does not happen — ChromaDB reads from disk.)

Query: "What are the main impacts of climate change on India's agriculture and food secu..."

  Rank  Cosine Sim  Visual                               Source
  ---- ----------  -----------------------------------  ----------------------------------------
  [1]   0.7459    [######################--------]  kipedia.org/wiki/Climate_change_in_India
  [2]   0.5817    [#################-------------]  kipedia.org/wiki/Climate_change_in_India
  [3]   0.5804    [#################-------------]  kipedia.org/wiki/Climate_change_in_India
  [4]   0.5728    [#################-------------]  kipedia.org/wiki/Climate_change_in_India
  [5]   0.5726    [#################-------------]  //en.wikipedia.org/wiki/Climate_of_India

Scores range fro

### What retrieval tells us

Look at the 5 chunks above. Notice:
- They come from **different sources** (MMR doing its job — diversity)
- They all relate to **agriculture, food, or climate impacts** (relevance doing its job)
- The LLM will answer *only from these 5 chunks* — nothing else

This is the single most important concept in RAG: **the answer quality is bounded by retrieval quality**. If the right chunk isn't in these 5, the LLM cannot give a correct answer — it will either say it doesn't know (good) or hallucinate (bad). We'll measure this in evals.

---
## Step 6 — Generation

**What happens here:** The 5 retrieved chunks are formatted into a context string and injected into the prompt alongside the question. The LLM reads only that context and writes an answer.

**The prompt instructs the model explicitly:**
> *"Answer using ONLY the context below. If the context doesn't contain enough information, say so — don't make things up."*

This instruction is what separates RAG from a vanilla LLM call. The model is grounded — it's not allowed to use its training data, only the retrieved passages.

**Mental model:** You're a student given 5 index cards and told: "Write your answer using only what's on these cards. Do not use your memory. Cite which cards you used." That's exactly what the LLM is doing.

In [9]:
from rag.generator import build_llm, get_rag_chain, format_docs

llm       = build_llm(provider=config.llm_provider, model_name=config.llm_model, api_key=config.openai_api_key)
rag_chain = get_rag_chain(llm=llm)

# Format the retrieved chunks into a single context string
context = format_docs(retrieved)

print('--- Context sent to LLM (formatted retrieved chunks) ---')
print(context[:800] + '\n... [truncated for display]')
print()
print('=' * 60)
print(f'QUESTION: {question}')
print('=' * 60)

answer = rag_chain.invoke({'question': question, 'context': context})

print('\nANSWER:')
print(answer)

print('\n--- Sources ---')
for i, doc in enumerate(retrieved, 1):
    source = doc.metadata.get('source', '?')[-55:]
    print(f'  [{i}] {source}')

--- Context sent to LLM (formatted retrieved chunks) ---
[1] Source: https://en.wikipedia.org/wiki/Climate_change_in_India
Reduced crop yields[edit]
Climate change in India will have disproportionate ramifications for the more than 400 million that make up India's poor community. This is because so many depend on natural resources for their food, shelter and income. Indian farmers are the first to feel the economic effects of climate change since 85% of Indian farmers have low financial resilience.[60] More than 56% of people in India work in agriculture, while many others earn their living in coastal areas.[61]

[2] Source: https://en.wikipedia.org/wiki/Climate_change_in_India
Mitigation[edit]
India is ranked high in Climate Change Performance Index 2021 
Greenhouse gas sinks[edit]
Land use, land-use change, and forestry absorbed 300 Mt of CO2e
... [truncated for display]

QUESTION: What are the main impacts of climate change on India's agriculture and food security?

ANSWER:
The main

---
## Putting it all together — Full Pipeline

The `RAGPipeline.ask()` method runs all 6 steps (steps 5–6 for each query, steps 1–4 are done once at startup) in a single call. Use it going forward.

In [10]:
from rag.pipeline import RAGPipeline

pipeline = RAGPipeline(config)
# The pipeline re-uses the existing chroma_db — no re-embedding needed

questions = [
    'What role does deforestation play in worsening climate change in India?',
    'How is India expanding its renewable energy capacity?',
    'Which Indian states are most vulnerable to climate change impacts?',
]

for q in questions:
    print('\n' + '=' * 70)
    pipeline.ask(q, show_sources=True)
    print()


Q: What role does deforestation play in worsening climate change in India?

Answer:
The context does not provide specific information on the role of deforestation in worsening climate change in India. Therefore, I cannot answer the question based on the provided context.

Sources (5 chunks retrieved):
  [1] https://en.wikipedia.org/wiki/Climate_change_in_India -> Mitigation[edit] India is ranked high in Climate Change Performance Index 2021  Greenhouse gas sinks[edit] Land use, lan...
  [2] https://en.wikipedia.org/wiki/Climate_change_in_India -> Reduced crop yields[edit] Climate change in India will have disproportionate ramifications for the more than 400 million...
  [3] https://en.wikipedia.org/wiki/Deforestation_in_India -> Dynamics[edit] Deforestation has several causes including colonization, agricultural expansion, firewood collection, tim...
  [4] https://en.wikipedia.org/wiki/Climate_change_in_India -> Health effects[edit] Air pollution[edit] Air pollution and climate change

---
## Try it yourself

Edit the question below and run the cell. The pipeline will retrieve relevant chunks from the 4 Wikipedia articles and generate a grounded answer.

**Good questions to try:**
- `'What is India's NDC commitment under the Paris Agreement?'`
- `'How does the Indian Ocean affect India's monsoon patterns?'`
- `'What are the economic costs of climate change for India?'`
- `'Which sectors are the largest emitters of greenhouse gases in India?'`

**Also try a question the corpus can't answer** (to see the model admit it doesn't know rather than hallucinate):
- `'What is the GDP of Bangladesh in 2024?'`

In [12]:
# Edit this question and run the cell
MY_QUESTION = 'What is India doing to reduce its carbon emissions?'

pipeline.ask(MY_QUESTION, show_sources=True)

Q: What is India doing to reduce its carbon emissions?

Answer:
India is progressively decoupling economic growth from greenhouse gas emissions, having reduced its emission intensity of GDP by 24% between 2005 and 2016. The country aims to reduce emission intensity by 20–25% from 2005 levels by 2020 and has set a target to reduce it by a third by 2030. India has adequate carbon neutral resources such as biomass, wind, solar, and hydro power to achieve net zero carbon emissions. Additionally, India is a global leader in renewable energy and has implemented various instruments like energy saving certificates and renewable energy certificates.

Sources (5 chunks retrieved):
  [1] https://en.wikipedia.org/wiki/Climate_change_in_India -> In its Biennial Update Report to the United Nations Framework Convention on Climate Change (UNFCCC) submitted in Februar...
  [2] https://en.wikipedia.org/wiki/Climate_change_in_India -> Carbon emission trading and pricing[edit] Carbon emission trading is y

'India is progressively decoupling economic growth from greenhouse gas emissions, having reduced its emission intensity of GDP by 24% between 2005 and 2016. The country aims to reduce emission intensity by 20–25% from 2005 levels by 2020 and has set a target to reduce it by a third by 2030. India has adequate carbon neutral resources such as biomass, wind, solar, and hydro power to achieve net zero carbon emissions. Additionally, India is a global leader in renewable energy and has implemented various instruments like energy saving certificates and renewable energy certificates.'

---
## What's next — Evals

You've now seen every step of the RAG pipeline. The natural next question is: **how do we know if it's working well?**

Keyword checks (`must_contain`) tell you whether a word appears — but not whether the answer is actually correct, faithful to the sources, or relevant to the question.

We'll build three proper eval metrics:

| Metric | Question it answers |
|---|---|
| **Retrieval Precision@K** | Did the retriever surface the right chunks? |
| **Faithfulness** | Is the answer grounded in what was retrieved, or did the model hallucinate? |
| **Answer Relevance** | Does the answer actually address the question asked? |

The last two use **LLM-as-judge** — a separate LLM call scores each answer. This is the industry-standard approach because quality judgements require understanding, not pattern matching.

---
## Step 7 — Evals

**The question we have not answered yet: is the pipeline actually working well?**

Running queries and reading answers by hand does not scale. We need automated, quantitative scores.

The three metrics we measure:

| Metric | What it measures | How |
|---|---|---|
| **Retrieval Precision@K** | Did the retriever surface relevant chunks? | Keyword match — no LLM needed |
| **Faithfulness** | Is the answer grounded in the context, or did the model hallucinate? | LLM-as-judge |
| **Answer Relevance** | Does the answer address the question? | LLM-as-judge |

**LLM-as-judge:** For faithfulness and relevance, we send a second LLM call with a scoring prompt.
The judge reads the question, context, and answer, then returns a score 0.0 to 1.0 with a one-sentence reason.
This is the industry-standard approach used by RAGAS, TruLens, and DeepEval.

**Golden dataset:** We hand-wrote 7 Q&A pairs covering different topics in the corpus ().
You write them by hand because you know what a good answer looks like.
Automated golden datasets create circular eval loops.

**Targets:**

| Metric | Target |
|---|---|
| Retrieval Precision@K | >= 0.75 |
| Faithfulness | >= 0.80 |
| Answer Relevance | >= 0.80 |

In [5]:
import importlib, os
from dotenv import load_dotenv
load_dotenv(override=True)

import evals.golden_dataset as _gd
importlib.reload(_gd)
from evals.golden_dataset import GOLDEN_DATASET

from rag.config import RAGConfig
from rag.pipeline import RAGPipeline
from rag.eval import run_evals
from rag.generator import build_llm

# Bootstrap config + pipeline from persisted chroma_db if the kernel was restarted.
try:
    _config = config
    _pipeline = pipeline
    print("Using pipeline from notebook session.")
except NameError:
    print("config/pipeline not in scope — bootstrapping from chroma_db on disk...")
    _config = RAGConfig(llm_provider="openai", embedding_provider="openai")
    _pipeline = RAGPipeline(_config)
    print(f"Pipeline ready. Chunks in index: {_pipeline.vectorstore._collection.count()}")

anthropic_key = os.environ.get("ANTHROPIC_API_KEY", "")
openai_key    = os.environ.get("OPENAI_API_KEY", "")

if anthropic_key:
    judge_llm   = build_llm(provider="anthropic", model_name="claude-haiku-4-5-20251001", api_key=anthropic_key)
    judge_label = "anthropic / claude-haiku-4-5-20251001"
else:
    judge_llm   = build_llm(provider="openai", model_name="gpt-5.4-mini", api_key=openai_key)
    judge_label = "openai / gpt-5.4-mini  ⚠ same family as generator"

adversarial_ids = {"out-of-corpus", "unanswerable", "hallucination"}
n_adversarial = sum(any(x in c["id"] for x in adversarial_ids) for c in GOLDEN_DATASET)
n_normal = len(GOLDEN_DATASET) - n_adversarial

print(f"\nGenerator:   {_config.llm_provider} / {_config.llm_model}")
print(f"Judge:       {judge_label}")
print(f"Cases:       {len(GOLDEN_DATASET)} ({n_normal} normal + {n_adversarial} adversarial)")
print()

summary = run_evals(
    pipeline=_pipeline,
    golden_dataset=GOLDEN_DATASET,
    judge_llm=judge_llm,
)

config/pipeline not in scope — bootstrapping from chroma_db on disk...
Pipeline ready. Chunks in index: 348

Generator:   openai / gpt-4o-mini
Judge:       anthropic / claude-haiku-4-5-20251001
Cases:       10 (7 normal + 3 adversarial)


=== RAG Eval Report ===

ID      Precision@K  Faithfulness  Relevance
---------------------------------------------
q1             0.80          0.90       0.90
q2             1.00          1.00       0.90
q3             0.60          1.00       0.75
q4             0.60          1.00       1.00
q5             0.80          1.00       0.90
q6             1.00          1.00       0.50
q7             0.60          1.00       0.70
q8-out-of-corpus         0.80          1.00       0.00
q9-unanswerable-states         0.40          1.00       1.00
q10-hallucination-bait         0.40          1.00       0.00
---------------------------------------------
AVG            0.70          0.99       0.67

Targets:  Precision >= 0.75 | Faithfulness >= 0.80 | Relevanc

### Reading the results

**Retrieval Precision@K** is the simplest: of the 5 chunks retrieved, how many contained words relevant to the question?
Score 0.8 = 4 out of 5 chunks were on-topic.

**Faithfulness** catches hallucination: did the model stick to the context, or add things that were not there?
A low score means the model is making things up.

**Answer Relevance** catches off-topic answers: even if the answer is faithful to context, does it address what was asked?
A model that says "climate change affects many things" to every question scores high on faithfulness but low on relevance.

**What to do if scores are low:**

| Low metric | Likely cause | Fix |
|---|---|---|
| Precision@K | Wrong chunks retrieved | Tune , try  vs , improve chunking |
| Faithfulness | Model hallucinating | Tighten system prompt, reduce  to avoid noisy context |
| Relevance | Answer off-topic | Check corpus coverage, add more relevant source documents |